# Resume Gap Agentic Demo

An agentic pipeline that reads a resume + target role, sources live job
postings, extracts required skills across them, and produces a **prioritized
gap report**.

The pipeline is role- and seniority-agnostic. This demo covers 3 different jobs roles with different seniority levels:

| Resume | Target role | Level |
|---|---|---|
| `senior-accountant-elegant-resume-example.pdf` | Accountant | senior |
| `software_engineer_resume.pdf` | Software Engineer | mid |
| `urban_design_resume.pdf` | Urban Designer | entry |

Every run uses a LangGraph ReAct **sourcing agent** + **LLM
critic** (powered by GPT-4o-mini) over **MCP tools** served by `mcp_server.py`, live Adzuna API
postings, and real LLM extraction.

The pipeline has two nested agentic loops:

```text
                                     ┌───── outer loop: critic wants more coverage ────┐
                                     ▼                                                 │
parse_resume → parse_target_role → source_postings → extract → aggregate → gap_diff → critic ──(ok)──► report
                                     │         ▲
                                     └─────────┘  inner ReAct loop: the agent calls MCP tools
                                                  (search_jobs, get_posting_details, ...) until satisfied
```

- **Inner loop** — the ReAct sourcing agent decides how to query, whether to
  reword the search, and which postings to keep (bounded by `agent_recursion_limit`).
- **Outer loop** — the LLM critic judges whether the analysis rests on enough
  coverage; if not, the graph loops back to sourcing with a refinement hint
  (bounded by `max_critic_loops`).

The logs below shown for `sourcing_agent.done` and `critic_agent.done` indicates if a decision loop has been triggered, i.e., the critic agent decides there are not enough job postings related to the role, to produce a good enough feedback for the input resume.

## Setup

Imports, logging, PDF resume loading (`pypdf` text-layer extraction — scanned
PDFs need OCR and are rejected with a clear error), and the shared
`run_gap_analysis()` runner used by the three analysis cells below.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from pypdf import PdfReader

from config import Config
from logging_conf import configure_logging
from graph.build import build_graph, build_checkpointer
from graph.state import GraphState
from reporting import render_emoji_summary, gaps_dataframe

configure_logging('INFO', colors=True)

_keys = Config()
HAVE_KEYS = bool(_keys.openai_api_key and _keys.adzuna_app_id and _keys.adzuna_app_key)
if not HAVE_KEYS:
    print('Missing API keys — the analysis cells below will be skipped. '
          'Set OPENAI_API_KEY, RGA_ADZUNA_APP_ID, RGA_ADZUNA_APP_KEY in .env.')


def load_resume(path):
    """Read a resume as text; .pdf goes through pypdf text-layer extraction."""
    path = Path(path)
    if path.suffix.lower() == '.pdf':
        text = '\n'.join(page.extract_text() or '' for page in PdfReader(path).pages)
        if not text.strip():
            raise ValueError(
                f'{path.name} has no extractable text (scanned/image PDF?) — needs OCR.'
            )
        return text
    return path.read_text(encoding='utf-8')


def run_gap_analysis(resume_path, target_role, level, location='Singapore'):
    if not HAVE_KEYS:
        print('Skipped (missing API keys).')
        return None
    resume_path = Path(resume_path)
    resume_text = load_resume(resume_path)
    print(f'Resume: {resume_path.name} ({len(resume_text)} chars) '
          f'-> target: {level} {target_role} in {location}')

    config = Config(
        mode='agentic',
        use_mock_source=False,      # MCP tools search Adzuna live
        use_mock_llm=False,         # real LLM extraction on real posting text
        llm_provider='openai',
        max_critic_loops=1,         # one reflect-and-retry keeps runtime/cost bounded
        agent_recursion_limit=50,   # live sources tempt the agent into more exploration
    )
    graph = build_graph(config, checkpointer=build_checkpointer(':memory:'))
    initial = GraphState(
        resume_text=resume_text, target_role=target_role, level=level, location=location
    )
    final = GraphState.model_validate(
        graph.invoke(
            initial, config={'configurable': {'thread_id': f'demo-{resume_path.stem}'}}
        )
    )

    print(f'\n{len(final.postings)} {level}-level postings analyzed:')
    display(pd.DataFrame(
        [{'title': p.title, 'company': p.company} for p in final.postings]
    ))
    print(render_emoji_summary(final.report))
    if final.report.gaps:
        display(
            gaps_dataframe(final.report)
            .style.format({'frequency': '{:.0%}'})
            .hide(axis='index')
        )
    return final

## Running the pipeline on sample CV for gap analysis

For roles which are in abundance in the current job market, it will be easy the sourcing agent to find those. And higher likelihood of the search agent bringing back postings relevant to the resume for the critic to make sound judgement and provide good feedback.

### Senior Accountant — senior-level gap analysis

In [2]:
accountant_final = run_gap_analysis(
    'data/senior-accountant-elegant-resume-example.pdf', 'Accountant', 'senior'
)

Resume: senior-accountant-elegant-resume-example.pdf (2292 chars) -> target: senior Accountant in Singapore
00:55:25 [info     ] mcp_tools_loaded               tools=['search_jobs', 'get_posting_details', 'normalize_skill_name', 'list_known_skills']
00:55:25 [info     ] parse_resume.start             chars=2292 llm=openai
00:55:30 [info     ] parse_resume.done              latency_ms=5640.5 skills=8 years_experience=12.0
00:55:30 [info     ] parse_target_role.start        role=Accountant
00:55:30 [info     ] parse_target_role.done         canonical_query='senior accountant' latency_ms=2.7
00:55:30 [info     ] sourcing_agent.start           looped=False role=Accountant tools=['search_jobs', 'get_posting_details', 'normalize_skill_name', 'list_known_skills']
00:57:37 [warning  ] sourcing_agent.parse_failed    error='Expecting value: line 1 column 1 (char 0)'
00:57:41 [info     ] sourcing_agent.done            agent_tool_calls=32 chosen=20 entry_level=20 latency_ms=130997.4 query_used='se

,title,company
0,Senior Accountant,Pathlight School / Autism Resource Centre
1,Senior Accountant,Exclusive Networks
2,Senior Accountant,StorHub Self Storage
3,Senior Accountant,Watsons Singapore
4,Accountant/Senior Accountant,Global Finance & Technology Network
5,Senior Accountant (Subsi),Maritime and Port Authority of Singapore
6,Senior Accountant (AFM),Maritime and Port Authority of Singapore
7,Senior Accounting Specialist,Welinktalent
8,Senior Accountant - Tangspac,Tangspac
9,Senior Accountant (Contract),HMH


Gap report for Accountant (senior)
Postings analyzed: 20  |  Matched skills: 1  |  Gaps: 59

🔴 CRITICAL (0)
   (none)

🟡 IMPORTANT (1)
   - accounting (35% of postings)  [weak: accounting]

🟢 NICE-TO-HAVE (58)
   - budgeting (15% of postings)
   - process improvement (15% of postings)
   - team collaboration (15% of postings)
   - tax filings (15% of postings)
   - internal controls (10% of postings)
   - compliance (10% of postings)
   - financial accounting (10% of postings)
   - financial analysis (10% of postings)
   - regulatory compliance (10% of postings)
   - cashflow planning (10% of postings)
   - statutory audit (10% of postings)
   - GST submissions (10% of postings)
   - XBRL filings (10% of postings)
   - preparation of Board papers (10% of postings)
   - financial operations (10% of postings)
   - compliance with Charities Act (5% of postings)
   - consolidation (5% of postings)
   - statutory financial statements (5% of postings)
   - system improvements (5% of postings

skill,category,frequency,severity,remediation
accounting,domain,35%,important,Take an introductory course or read up on accounting and note it in your summary.
budgeting,domain,15%,nice-to-have,Take an introductory course or read up on budgeting and note it in your summary.
process improvement,domain,15%,nice-to-have,Take an introductory course or read up on process improvement and note it in your summary.
team collaboration,soft,15%,nice-to-have,Highlight concrete examples that demonstrate team collaboration in your experience bullets.
tax filings,domain,15%,nice-to-have,Take an introductory course or read up on tax filings and note it in your summary.
internal controls,domain,10%,nice-to-have,Take an introductory course or read up on internal controls and note it in your summary.
compliance,domain,10%,nice-to-have,Take an introductory course or read up on compliance and note it in your summary.
financial accounting,domain,10%,nice-to-have,Take an introductory course or read up on financial accounting and note it in your summary.
financial analysis,domain,10%,nice-to-have,Take an introductory course or read up on financial analysis and note it in your summary.
regulatory compliance,domain,10%,nice-to-have,Take an introductory course or read up on regulatory compliance and note it in your summary.


### Software Engineer — mid-level gap analysis

In [3]:
engineer_final = run_gap_analysis(
    'data/software_engineer_resume.pdf', 'Software Engineer', 'mid'
)

Resume: software_engineer_resume.pdf (1702 chars) -> target: mid Software Engineer in Singapore
00:59:54 [info     ] mcp_tools_loaded               tools=['search_jobs', 'get_posting_details', 'normalize_skill_name', 'list_known_skills']
00:59:54 [info     ] parse_resume.start             chars=1702 llm=openai
00:59:57 [info     ] parse_resume.done              latency_ms=2546.4 skills=23 years_experience=4.0
00:59:57 [info     ] parse_target_role.start        role='Software Engineer'
00:59:57 [info     ] parse_target_role.done         canonical_query='mid software engineer' latency_ms=6.9
00:59:57 [info     ] sourcing_agent.start           looped=False role='Software Engineer' tools=['search_jobs', 'get_posting_details', 'normalize_skill_name', 'list_known_skills']
01:00:44 [info     ] sourcing_agent.done            agent_tool_calls=4 chosen=40 entry_level=27 latency_ms=47479.4 query_used='Software Engineer' reasoning='The search returned a sufficient number of relevant mid-level soft

,title,company
0,Software Engineer,Applied Materials
1,Software Engineer,Fuku
2,Software Engineer,Fuku
3,Software engineer,Fuku
4,Software Engineer,Fuku
5,Software Engineer,YouTrip
6,Software Engineer,Fuku
7,Software Engineer,LRDTECH
8,Software Engineer,Fuku
9,Software Engineer,Newbridge


Gap report for Software Engineer (mid)
Postings analyzed: 17  |  Matched skills: 2  |  Gaps: 43

🔴 CRITICAL (0)
   (none)

🟡 IMPORTANT (2)
   - software development (47% of postings)
   - collaboration (41% of postings)

🟢 NICE-TO-HAVE (41)
   - troubleshooting (29% of postings)
   - clean code (24% of postings)
   - debugging (24% of postings)
   - emerging technologies (12% of postings)
   - code reviews (12% of postings)
   - OCaml (12% of postings)
   - functional programming (12% of postings)
   - programming languages (6% of postings)
   - continuous improvement (6% of postings)
   - data center management (6% of postings)
   - scalable code (6% of postings)
   - efficient code (6% of postings)
   - software development processes (6% of postings)
   - code quality (6% of postings)
   - industry best practices (6% of postings)
   - Java (6% of postings)
   - backend development (6% of postings)
   - software design (6% of postings)
   - Software Engineering (6% of postings)
   - A

skill,category,frequency,severity,remediation
software development,domain,47%,important,Take an introductory course or read up on software development and note it in your summary.
collaboration,soft,41%,important,Highlight concrete examples that demonstrate collaboration in your experience bullets.
troubleshooting,technical,29%,nice-to-have,Build a small portfolio project that uses troubleshooting; add it to your resume.
clean code,technical,24%,nice-to-have,Build a small portfolio project that uses clean code; add it to your resume.
debugging,technical,24%,nice-to-have,Build a small portfolio project that uses debugging; add it to your resume.
emerging technologies,domain,12%,nice-to-have,Take an introductory course or read up on emerging technologies and note it in your summary.
code reviews,technical,12%,nice-to-have,Build a small portfolio project that uses code reviews; add it to your resume.
OCaml,technical,12%,nice-to-have,Build a small portfolio project that uses OCaml; add it to your resume.
functional programming,domain,12%,nice-to-have,Take an introductory course or read up on functional programming and note it in your summary.
programming languages,technical,6%,nice-to-have,Build a small portfolio project that uses programming languages; add it to your resume.


## Urban Designer — entry-level gap analysis

Deliberately picked a more niche job role in Singapore to have a higher chance of the agentic pipeline going into a loop due to limited job listings of urban desgin roles in SG.

In [4]:
designer_final = run_gap_analysis(
    'data/urban_design_resume.pdf', 'Urban Designer', 'entry'
)

Resume: urban_design_resume.pdf (1634 chars) -> target: entry Urban Designer in Singapore


01:04:33 [info     ] mcp_tools_loaded               tools=['search_jobs', 'get_posting_details', 'normalize_skill_name', 'list_known_skills']
01:04:33 [info     ] parse_resume.start             chars=1634 llm=openai
01:04:36 [info     ] parse_resume.done              latency_ms=3159.2 skills=20 years_experience=0.0
01:04:36 [info     ] parse_target_role.start        role='Urban Designer'
01:04:36 [info     ] parse_target_role.done         canonical_query='entry urban designer' latency_ms=18.2
01:04:36 [info     ] sourcing_agent.start           looped=False role='Urban Designer' tools=['search_jobs', 'get_posting_details', 'normalize_skill_name', 'list_known_skills']
01:06:11 [warning  ] sourcing_agent.parse_failed    error='Expecting value: line 1 column 1 (char 0)'
01:06:16 [info     ] sourcing_agent.done            agent_tool_calls=25 chosen=20 entry_level=6 latency_ms=99248.4 query_used='entry Urban Designer' reasoning='fallback: used plain search after unparseable agent output' re

,title,company
0,"Trainee, Landscape Architecture",DP Green
1,Interior Designer,Aedas
2,Planner,WATG
3,Resident Architect,Aedas
4,Site Document Controller,Aedas
5,Resident Technical Officer,Aedas


Gap report for Urban Designer (entry)
Postings analyzed: 6  |  Matched skills: 0  |  Gaps: 5

🔴 CRITICAL (0)
   (none)

🟡 IMPORTANT (0)
   (none)

🟢 NICE-TO-HAVE (5)
   - interior design (17% of postings)
   - sustainability (17% of postings)
   - architecture (17% of postings)  [weak: architecture]
   - design software (17% of postings)
   - communication (17% of postings)


skill,category,frequency,severity,remediation
interior design,domain,17%,nice-to-have,Take an introductory course or read up on interior design and note it in your summary.
sustainability,domain,17%,nice-to-have,Take an introductory course or read up on sustainability and note it in your summary.
architecture,domain,17%,nice-to-have,Take an introductory course or read up on architecture and note it in your summary.
design software,technical,17%,nice-to-have,Build a small portfolio project that uses design software; add it to your resume.
communication,soft,17%,nice-to-have,Highlight concrete examples that demonstrate communication in your experience bullets.
